# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [13]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

#file_path = "../docs/Managing_Oneself_Drucker_HBR.pdf"

#loaded the file from the URL instead of the downloaded pdf file

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(url)

docs = loader.load()

#print(len(docs)) #used for testing

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

#print(document_text) #used for testing
#text stored in document_text


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [46]:
from openai import OpenAI
from pydantic import BaseModel
import os

class ArticleExtract(BaseModel):
    Title: str
    Author: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})



#user specified tone. this could be modified to ask the user for the tone in the program

tone  = "Victorian English"

#set dev / system prompt

system_prompt = """
    
    The user is new to AI and they are trying to understand why this article is relevant for an AI professional in their professional development.

    To assist the user, you will extract text from the article downloaded in a structured format
    
    The requirements are:

    1. Identify the book's Title
    2. Identify the book's Author
    3. Identify Relevance by creating a statement, no longer than one paragraph, that explains why is this article is relevant for an AI professional in their professional development.
    4. Create a Summary that gives a concise and succinct summary of the text no longer than 1000 tokens. The summary should use the tone specified by the user.

    Provide your response in the structured format matching the structured format specified exactly
"""

#set user prompt

user_prompt = f"""
Analyze the AI article for my professional development and return structured output

The text from the pdf is located here: 
    
    <article>
    {document_text}
    </article>
Please use this tone for the summary: {tone}    
"""
response = client.responses.parse(
    #model = 'gpt-4o',
    model = "gpt-4o-mini", # depending on the tier we have available, we might need to update the model to be used
    input =[
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleExtract
)

#assign the response to struct_output

struct_output = response.output_parsed

#we now have the structured output in struct_output

# get the # of input and output tokens. add to output
struct_output.InputTokens = response.usage.input_tokens
struct_output.OutputTokens = response.usage.output_tokens

#display the response

from IPython.display import display, Markdown

display(Markdown(f"""
 Title: {struct_output.Title}
 
 Author: {struct_output.Author}

 Relevance:
 {struct_output.Relevance}

 Summary:
 {struct_output.Summary}
 """))   

#display the specified tone and the input/output tokens based on the response object

print(f"InputTokens: {struct_output.InputTokens}")
print(f"OutputTokens: {struct_output.OutputTokens}")



 Title: Managing Oneself

 Author: Peter F. Drucker

 Relevance:
 This article is highly relevant for AI professionals seeking to navigate the complexities of their careers in an evolving technological landscape. By understanding their strengths, values, and optimal work styles, AI practitioners can manage their career trajectories more effectively and align their personal goals with organizational expectations, fostering both personal and professional growth.

 Summary:
 In this illuminating discourse, the esteemed Peter F. Drucker expounds the necessity of self-management in the contemporary knowledge economy. He elucidates that success within this realm is not merely a product of ambition and intelligence, but rather a profound understanding of oneself. Ambitious individuals are called upon to assume the role of their own chief executive officers, thereby assuming greater responsibility for their career paths. By engaging in practices such as feedback analysis, wherein one records anticipated outcomes against actual results, individuals cultivate a clearer comprehension of their intrinsic strengths and weaknesses. Drucker urges a focus on personal values, emphasizing the significance of ensuring that one's ethical beliefs resonate harmoniously with those of their organization. Furthermore, he posits that understanding one's optimal work environment and collaborative preferences is crucial for achieving exceptional performance. The treatise ultimately advocates for a proactive approach to career development, wherein individuals articulate their contributions and align their professional endeavors with their innate capabilities and moral compass.
 

InputTokens: 12455
OutputTokens: 283


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [47]:
from pydantic import BaseModel
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel

class EvalOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

#Use the original document text and the structured output summary

test_case = LLMTestCase(input=document_text, actual_output=struct_output.Summary)

# Summarization
Summarization = SummarizationMetric(
    threshold=0.5,
    assessment_questions=[
        "Does the summary capture the main points and essential information from the original content",
        "Is the summary concise as per the instructions?",
        "Is the summary well-structured and readable",
        "Does this summary repeat the same point twice in different words?",
        "Is the summary factually correct?"
    ],
    model=model
)
# Coherence
Clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Is it logically structured?",
        "Are ideas connected clearly?",
        "Is it easy to understand?",
        "Any ambiguity?",
        "Any redundancy?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Tonality
Tonality = GEval(
    name="Tonality",
    evaluation_steps=[
        "Is the tone consistent?",
        "Is the tone clearly identifiable?",
        "Does it match the requested tone?",
        "Does it improve readability?",
        "Is it appropriate professionally?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Safety
Safety = GEval(
    name="Safety",
    evaluation_steps=[
        "Is it free from harmful language?",
        "Is there misinformation?",
        "Is there bias?",
        "Is content unsafe?",
        "Is it professional?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)


Summarization.measure(test_case)
Clarity.measure(test_case)
Tonality.measure(test_case)
Safety.measure(test_case)

#from IPython.display import display, Markdown
#display(Markdown(f'**Score**: {Summarization.score}'))
#display(Markdown(f'**Reason**: {Summarization.reason}'))

#create an evaloutput object and assign the results
Eval_result = EvalOutput(
    SummarizationScore=Summarization.score,
    SummarizationReason=Summarization.reason,
    CoherenceScore=Clarity.score,
    CoherenceReason=Clarity.reason,
    TonalityScore=Tonality.score,
    TonalityReason=Tonality.reason,
    SafetyScore=Safety.score,
    SafetyReason=Safety.reason
)

#print the results in a structured format

print("\n Evaluation Results")

print(Eval_result.model_dump_json(indent=2))


Output()

Output()

Output()

Output()


 Evaluation Results
{
  "SummarizationScore": 0.9090909090909091,
  "SummarizationReason": "The score is 0.91 because the summary accurately reflects the main points of the original text without any contradictions, although it introduces extra information regarding ethical beliefs that was not present in the original text.",
  "CoherenceScore": 0.8720382847128485,
  "CoherenceReason": "The response is logically structured and presents ideas clearly, effectively summarizing Drucker's key points on self-management and personal responsibility in the knowledge economy. It connects concepts such as feedback analysis, personal values, and optimal work environments without ambiguity. The language is accessible, making it easy to understand, and there is minimal redundancy. However, a slight improvement could be made by providing more specific examples from the text to enhance clarity further.",
  "TonalityScore": 0.9012673506695551,
  "TonalityReason": "The response maintains a consistent an

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
#As per the evaluation, the summarization metric score was 0.0
#The evaluation reported that the summary contained extra information



#We will update the instructions to ensure there is no extra information
#We will be more strict so there is less deviation from original content.

new_system_prompt = """
    
    The user is new to AI and they are trying to understand why this article is relevant for an AI professional in their professional development.

    To assist the user, you will extract text from the article downloaded in a structured format
    
    The requirements are:

    1. Identify the book's Title
    2. Identify the book's Author
    3. Identify Relevance by creating a statement, no longer than one paragraph, that explains why is this article is relevant for an AI professional in their professional development.
    4. Create a Summary that gives a concise and succinct summary of the text no longer than 1000 tokens. The summary should use the tone specified by the user. 
    5. Do not include extra information about optimal work environments 
    6. do not deviate from original content.

    Provide your response in the structured format matching the structured format specified exactly
"""

#use the new system prompt based on the evaluation.

response = client.responses.parse(
    #model = 'gpt-4o',
    model = "gpt-4o-mini", # depending on the tier we have available, we might need to update the model to be used
    input =[
    {"role": "developer", "content": new_system_prompt},
    {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleExtract
)

new_struct_output = response.output_parsed

#display the updated summary
#display the response

from IPython.display import display, Markdown

display(Markdown(f"""
 
 Updated Summary:
 {new_struct_output.Summary}
 """))   

# we now re-do the evaluation - on the new summary. new struct output.
# use a new test case 2 to keep it separate

test_case2 = LLMTestCase(input=document_text, actual_output=new_struct_output.Summary)

Summarization.measure(test_case2)
Clarity.measure(test_case2)
Tonality.measure(test_case2)
Safety.measure(test_case2)

#from IPython.display import display, Markdown
#display(Markdown(f'**Score**: {Summarization.score}'))
#display(Markdown(f'**Reason**: {Summarization.reason}'))

#create an evaloutput object and assign the results
New_Eval_result = EvalOutput(
    SummarizationScore=Summarization.score,
    SummarizationReason=Summarization.reason,
    CoherenceScore=Clarity.score,
    CoherenceReason=Clarity.reason,
    TonalityScore=Tonality.score,
    TonalityReason=Tonality.reason,
    SafetyScore=Safety.score,
    SafetyReason=Safety.reason
)

#print the results in a structured format

print("\n Evaluation Results")

print(New_Eval_result.model_dump_json(indent=2))

#We did get a slightly higher score based on the eveluation.


# the score increased from 0.9090 to about 0.92

# based on the evaluation, I updated the prompt to not include extra information and not to deviate from the original content.

# It was obviously better because I was able to respond to the evaluatio and update the prompt.
# I do not believe this could be enough. The model evaluation could detect another issue and you may have to respond or update each time.

#however the key learning here is that its really up to your assessment questions
#if your assessment questions are too strict or are unclear, then evaluation doesnt work correctly.
#Originally I was getting 0.0 with the original summary.
#But I realized my assessment questions were confusing. The system was misunderstanding them.
#I made my assessment questions more clear and less confusing. The system was able to do a proper evaluation and not hallucinate with the evaluation.
#So all in all these controls can work - but you need to have the right assessment to ensure you are evaluating correctly.







 Updated Summary:
 In this esteemed treatise, Peter F. Drucker expounds upon the vital necessity of self-management within the contemporary knowledge economy. He posits that in an era rife with unparalleled opportunity, the onus of career navigation falls upon the individual, thereby designating oneself as the chief executive officer of one's professional journey. Central to this endeavor is a profound self-awareness, encompassing one's strengths, weaknesses, learning styles, and intrinsic values. To delineate one’s strengths, Drucker advocates the practice of feedback analysis, through which individuals may discern their capabilities by juxtaposing anticipated outcomes with actual results over time. He implores the reader to eschew the futile pursuit of ameliorating areas of deficiency, instead suggesting a concerted effort in cultivating one's inherent talents for optimal performance. Furthermore, the author elucidates the significance of aligning personal and organizational values to avert existential frustrations within the workplace. In contemplating one’s rightful domain, he encourages individuals to judiciously ascertain the environments wherein they thrive, ultimately transforming ordinary achievements into extraordinary contributions. The treatise concludes with a discourse on the imperative of embracing responsibilities in interpersonal relationships, underscoring the need for effective communication and mutual understanding within professional encounters. Thus, in managing one’s career, individuals emerge not merely as passive participants but as proactive architects of their destiny, capable of eliciting substantive impact in their respective fields.
 

Output()

Output()

Output()

Output()


 Evaluation Results
{
  "SummarizationScore": 0.9230769230769231,
  "SummarizationReason": "The score is 0.92 because the summary effectively captures the main points of the original text, despite introducing extra information about existential frustrations in the workplace that was not present in the original. This additional detail, while not contradictory, slightly detracts from the overall accuracy of the summary.",
  "CoherenceScore": 0.846727818398817,
  "CoherenceReason": "The response is logically structured and presents ideas in a clear and connected manner, effectively summarizing Drucker's key points on self-management. It is easy to understand, with a coherent flow from the importance of self-awareness to the need for effective communication in professional relationships. However, there is some redundancy in the phrasing, particularly in reiterating the importance of aligning personal and organizational values, which could have been more succinctly expressed.",
  "Tonality

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
